# Episode Visualization

Visualize how free energy changes across epochs for the same episode.

In [220]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Global figure font: Times New Roman (fallback to serif)
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",  # Keep equation text in serif style
    }
)

# BASE_DIR = ".../log/robot_mirror/rep5easy/eval_period500/nodecay/1-0/"
# STICKER_DETACHED_STEP = 15  # seed 1 = 15, seed 2 = 13

BASE_DIR = ".../log/robot_mirror/rep5easy/eval_period500/nodecay/2-0/"
STICKER_DETACHED_STEP = 13  # seed 1 = 15, seed 2 = 13

recorded_episode = 50000
EPISODE_IDX = 0  # Replay CSV may contain multiple episodes; select one here.
# checkpoint_episodes = [500, 10000, 20000, 30000, 40000, 50000]
checkpoint_episodes = [0, 50000]
MAX_PLOT_STEPS = 25  # Plot only this number of steps.

# Figure aspect ratio: "4:3" or "1:1" (width:height)
FIGURE_WIDTH_IN = 3  # Width in inches; height follows the selected aspect ratio automatically
_fig_h = FIGURE_WIDTH_IN  # 1:1
# _fig_h = FIGURE_WIDTH_IN * 3/4  # 4:3
# _fig_h = FIGURE_WIDTH_IN * 9/16  # 16:9
# _fig_h = FIGURE_WIDTH_IN * 16/9  # 9:16

In [221]:
# Read CSVs for all checkpoint_episodes and plot the selected episode
# (only the first MAX_PLOT_STEPS are shown)

import csv
from matplotlib.ticker import MultipleLocator

def compute_segment_mean(step_efe_pairs):
    if not step_efe_pairs:
        return None
    return float(np.mean([efe for _, efe in step_efe_pairs]))


def format_train_steps_compact(n):
    """Format large train-step values compactly (e.g., 500000 -> 500k, 1500000 -> 1.5M)."""
    n = int(round(n))
    if n >= 1_000_000:
        if n % 1_000_000 == 0:
            return f"{n // 1_000_000}M"
        x = n / 1_000_000
        s = f"{x:.1f}".rstrip("0").rstrip(".")
        return f"{s}M"
    if n >= 1_000:
        if n % 1_000 == 0:
            return f"{n // 1_000}k"
        x = n / 1_000
        s = f"{x:.1f}".rstrip("0").rstrip(".")
        return f"{s}k"
    return str(n)


def load_episode_efe_series(base_dir, checkpoint_episode, recorded_episode, episode_idx=0):
    csv_path = os.path.join(
        base_dir, f"replay_efe_checkpoint_{checkpoint_episode}_{recorded_episode}.csv"
    )
    if os.path.isfile(csv_path):
        steps = []
        efes = []
        with open(csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames or []
            has_episode_idx = "episode_idx" in fieldnames
            for row in reader:
                if has_episode_idx and int(row["episode_idx"]) != int(episode_idx):
                    continue
                steps.append(int(row["step"]))
                efes.append(float(row["efe"]))
        if steps:
            return steps, efes

    legacy_csv_path = os.path.join(
        base_dir,
        f"replay_episode_{episode_idx}_efe_checkpoint_{checkpoint_episode}_{recorded_episode}.csv",
    )
    if os.path.isfile(legacy_csv_path):
        steps = []
        efes = []
        with open(legacy_csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                steps.append(int(row["step"]))
                efes.append(float(row["efe"]))
        if steps:
            return steps, efes

    print(f"Missing replay CSV for episode_idx={episode_idx}: {csv_path}")
    return None, None


efe_values_by_ckpt = {}
for ckpt in checkpoint_episodes:
    steps, efes = load_episode_efe_series(
        BASE_DIR,
        ckpt,
        recorded_episode,
        episode_idx=EPISODE_IDX,
    )
    if steps is None or efes is None:
        continue

    steps = steps[:MAX_PLOT_STEPS]
    efes = efes[:MAX_PLOT_STEPS]
    efe_values_by_ckpt[ckpt] = (steps, efes)

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, _fig_h))
for ckpt, (steps, efes) in efe_values_by_ckpt.items():
    pre_pairs = [(step, efe) for step, efe in zip(steps, efes) if step < STICKER_DETACHED_STEP]
    post_pairs = [(step, efe) for step, efe in zip(steps, efes) if step >= STICKER_DETACHED_STEP]

    pre_mean = compute_segment_mean(pre_pairs)
    post_mean = compute_segment_mean(post_pairs)

    epoch_per_loop = 100
    collect_per_loop = 10
    train_steps = int((ckpt/collect_per_loop)*epoch_per_loop)
    label = f"{format_train_steps_compact(train_steps)} train steps"
    if pre_mean is not None and post_mean is not None:
        delta = post_mean - pre_mean
        label += f" (Δ={delta:+.2f})"

    if train_steps == 0:
        plot_kwargs = {"color": "tab:blue", "linestyle": "--", "linewidth": 2.5}
    elif train_steps == 500_000:
        plot_kwargs = {"color": "tab:orange", "linestyle": "-", "linewidth": 2.5}
    else:
        plot_kwargs = {"linestyle": "-", "linewidth": 1.5}

    line, = ax.plot(steps, efes, label=label, **plot_kwargs)
    color = line.get_color()
    mean_line_kw = {"colors": color, "linestyles": "-", "linewidth": 1, "alpha": 0.85}

    if pre_pairs:
        pre_steps = [step for step, _ in pre_pairs]
        # Extend the left-segment mean line up to the sticker detach step.
        ax.hlines(pre_mean, min(pre_steps), STICKER_DETACHED_STEP, **mean_line_kw)

    if post_pairs:
        post_steps = [step for step, _ in post_pairs]
        ax.hlines(post_mean, min(post_steps), max(post_steps), **mean_line_kw)

ax.axvline(STICKER_DETACHED_STEP, color="gray", linestyle="-.", linewidth=1.5, alpha=0.8)
ymin, ymax = ax.get_ylim()
ax.set_ylim(bottom=ymin * 0.5, top=ymax)
ax.text(
    STICKER_DETACHED_STEP,
    ymax - 0.3 * (ymax - ymin),
    "→ Detached \n    (Step 13)",
    ha="left",
    va="top",
    fontsize=12,
    color="dimgray",
)
ax.set_xlabel("Simulation step (seed 2)")
ax.set_ylabel("Expected Free Energy (EFE)")
ax.set_title(f"EFE in single episode")
if efe_values_by_ckpt:
    ax.legend(loc="lower right", fontsize=9)
ax.xaxis.set_major_locator(MultipleLocator(5))
out_stem = os.path.join(
    BASE_DIR,
    f"efe_plot_ep{recorded_episode}_episodeidx{EPISODE_IDX}_steps{MAX_PLOT_STEPS}_detach{STICKER_DETACHED_STEP}",
)
fig.savefig(out_stem + ".png", dpi=300, bbox_inches="tight")
fig.savefig(out_stem + ".pdf", bbox_inches="tight")
print(f"{out_stem}.png")
print(f"{out_stem}.pdf")
plt.show()